In [1]:
import time
import os
import json
import logging

from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

# Настройка логирования для отслеживания процесса
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Константы ---
# Главная страница каталога, с которой мы начинаем
CATALOG_BASE_URL = "https://5ka.ru/catalog"

# Пути для сохранения данных
EXTERNAL_DATA_PATH = "../../data/external"
CATEGORIES_URLS_FILE = os.path.join(EXTERNAL_DATA_PATH, "categories_urls.json")

# Создаем директорию, если она не существует
os.makedirs(EXTERNAL_DATA_PATH, exist_ok=True)

In [2]:
def scrape_category_urls(max_retries=3):
    """
    Основная функция для сбора URL категорий с механизмом повторных попыток.
    """
    for attempt in range(max_retries):
        logging.info(f"Попытка {attempt + 1} из {max_retries}...")
        driver = None
        try:
            # --- Инициализация драйвера ---
            options = webdriver.ChromeOptions()
            options.add_argument('--headless')
            options.add_argument('--no-sandbox')
            options.add_argument('--disable-dev-shm-usage')
            options.add_argument('window-size=1920x1080')
            options.add_argument(
                "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

            service = ChromeService(ChromeDriverManager().install())
            driver = webdriver.Chrome(service=service, options=options)
            driver.set_page_load_timeout(60)  # Увеличиваем таймаут загрузки страницы
            driver.implicitly_wait(10)  # Неявное ожидание

            # --- Процесс сбора ---
            logging.info(f"Открытие страницы: {CATALOG_BASE_URL}")
            driver.get(CATALOG_BASE_URL)

            wait = WebDriverWait(driver, 45)  # Увеличиваем таймаут ожидания элемента
            # Ждем не просто появления, а видимости элемента
            wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "[data-qa^='section-category-item-']")))

            logging.info("Страница каталога загружена, начинаем парсинг HTML...")
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')

            category_links = soup.select("a[data-qa^='section-category-item-']")

            if not category_links:
                logging.warning("Ссылки на категории не найдены, хотя элемент присутствовал. Повторная попытка...")
                raise ValueError("Пустой список ссылок на категории")

            category_urls = [link['href'] for link in category_links if link.has_attr('href')]

            unique_category_urls = sorted(list(set(category_urls)))

            logging.info(f"Сбор успешно завершен на попытке {attempt + 1}.")
            return unique_category_urls

        except Exception as e:
            logging.error(f"Ошибка на попытке {attempt + 1}: {e}")
            if attempt < max_retries - 1:
                logging.info("Ожидание 5 секунд перед следующей попыткой...")
                time.sleep(5)
            else:
                logging.error("Все попытки исчерпаны. Сбор не удался.")
                return []
        finally:
            if driver:
                driver.quit()

In [3]:
final_urls = scrape_category_urls()

if final_urls:
    logging.info(f"Итого: найдено {len(final_urls)} уникальных URL категорий.")
    print("Примеры найденных URL:")
    print("\n".join(final_urls[:5]))

    try:
        with open(CATEGORIES_URLS_FILE, 'w', encoding='utf-8') as f:
            json.dump(final_urls, f, ensure_ascii=False, indent=2)
        logging.info(f"URL-адреса категорий успешно сохранены в файл: {CATEGORIES_URLS_FILE}")
    except IOError as e:
        logging.error(f"Ошибка при сохранении файла: {e}")
else:
    logging.warning("Не удалось собрать URL-адреса. Файл не создан.")

2025-09-21 13:21:14,361 - INFO - Попытка 1 из 3...
2025-09-21 13:21:14,361 - INFO - ====== WebDriver manager ======
2025-09-21 13:21:15,573 - INFO - Get LATEST chromedriver version for google-chrome
2025-09-21 13:21:16,203 - INFO - Get LATEST chromedriver version for google-chrome
2025-09-21 13:21:16,867 - INFO - Get LATEST chromedriver version for google-chrome
2025-09-21 13:21:18,318 - INFO - WebDriver version 140.0.7339.185 selected
2025-09-21 13:21:18,325 - INFO - Modern chrome version https://storage.googleapis.com/chrome-for-testing-public/140.0.7339.185/win32/chromedriver-win32.zip
2025-09-21 13:21:18,326 - INFO - About to download new driver from https://storage.googleapis.com/chrome-for-testing-public/140.0.7339.185/win32/chromedriver-win32.zip
2025-09-21 13:21:18,941 - INFO - Driver downloading response is 200
2025-09-21 13:21:20,362 - INFO - Get LATEST chromedriver version for google-chrome
2025-09-21 13:21:22,127 - INFO - Driver has been saved in cache [C:\Users\petro\.wdm\

Примеры найденных URL:
/catalog/apteka--251C13027/
/catalog/aromaty-dlya-doma--251C13022/
/catalog/bez-laktozy-khalyal--251C13041/
/catalog/bumaga-i-salfetki--251C13026/
/catalog/chay-kofe-kakao--251C13057/


In [1]:
import time
import os
import json
import logging
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm

# Настройка логирования для отслеживания процесса
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Константы ---
BASE_URL = "https://5ka.ru"
EXTERNAL_DATA_PATH = "../../data/external"
CATEGORIES_URLS_FILE = os.path.join(EXTERNAL_DATA_PATH, "categories_urls.json")
RAW_PRODUCTS_FILE = os.path.join(EXTERNAL_DATA_PATH, "products_5ka_raw.json")

In [2]:
def connect_to_existing_chrome():
    """
    Подключается к уже запущенному экземпляру Chrome.
    """
    logging.info("Попытка подключения к существующему экземпляру Chrome на порту 9222...")
    try:
        options = Options()
        options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
        driver = webdriver.Chrome(options=options)
        logging.info("Успешно подключено к существующему браузеру.")
        return driver
    except Exception as e:
        logging.error(
            f"Не удалось подключиться к браузеру. Убедитесь, что он запущен с флагом --remote-debugging-port=9222. Ошибка: {e}")
        return None

In [3]:
def parse_products_from_page(soup: BeautifulSoup, category_url: str, category_name: str) -> list:
    """
    Парсит все карточки товаров с HTML-кода одной страницы, добавляя информацию о категории.
    """
    products_on_page = []
    product_cards = soup.select(".productCard_container__7EuBL")

    for card in product_cards:
        try:
            name_tag = card.select_one(".mainInformation_title__ziiEa")
            name = name_tag.text.strip() if name_tag else None

            weight_tag = card.select_one(".mainInformation_weight__o6cXn")
            weight = weight_tag.text.strip() if weight_tag else None

            price_tag = card.select_one(".priceContainer_price__AY8C_")
            price = price_tag['content'] if price_tag and price_tag.has_attr('content') else None

            link_tag = card.select_one("a.productCard_insteadContainer__DuFrN")
            link = link_tag['href'] if link_tag and link_tag.has_attr('href') else None

            if name:
                products_on_page.append({
                    "name": name,
                    "weight": weight,
                    "price": price,
                    "link": link,
                    "category_url": category_url,
                    "category_name": category_name
                })
        except Exception as e:
            logging.warning(f"Не удалось распарсить карточку товара: {e}")
            continue

    return products_on_page

In [4]:
# --- Загрузка URL ---
try:
    with open(CATEGORIES_URLS_FILE, 'r', encoding='utf-8') as f:
        category_urls = json.load(f)
    logging.info(f"Успешно загружено {len(category_urls)} URL категорий из файла.")
except Exception as e:
    logging.error(f"Не удалось загрузить URL категорий: {e}")
    category_urls = []

# --- Подключаемся к браузеру ---
driver = connect_to_existing_chrome()

# --- Основной цикл сбора ---
all_products = []
if driver and category_urls:
    # Загружаем прогресс, если он есть
    if os.path.exists(RAW_PRODUCTS_FILE):
        try:
            with open(RAW_PRODUCTS_FILE, 'r', encoding='utf-8') as f:
                all_products = json.load(f)
            logging.info(f"Обнаружен существующий файл. Загружено {len(all_products)} товаров.")
        except Exception:
            all_products = []

    # Собираем уже обработанные категории, чтобы их пропускать
    processed_categories = {p.get('category_url') for p in all_products}

    try:
        for category_url in tqdm(category_urls, desc="Обход категорий"):
            if category_url in processed_categories:
                logging.info(f"Категория {category_url} уже обработана. Пропускаем.")
                continue

            current_url = f"{BASE_URL}{category_url}"
            driver.get(current_url)

            try:
                # Первая попытка дождаться контента
                WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, ".productCard_container__7EuBL"))
                )

            except TimeoutException:
                # Если контент не появился, просим помощи
                print("-" * 50)
                print(f"!!! ВНИМАНИЕ: Возможна CAPTCHA на странице {current_url}")
                print("Пожалуйста, переключитесь в окно браузера, решите проверку (например, поставьте картинку).")
                input("ПОСЛЕ РЕШЕНИЯ НАЖМИТЕ ENTER В ЭТОЙ КОНСОЛИ, ЧТОБЫ ПРОДОЛЖИТЬ...")
                print("Спасибо! Продолжаем работу...")
                # После вмешательства, даем странице еще шанс
                time.sleep(5)

            # После ожидания или ручного вмешательства, запускаем скроллинг
            logging.info(f"Начинаем полную прокрутку для {current_url}...")
            last_height = driver.execute_script("return document.body.scrollHeight")
            while True:
                driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                time.sleep(2.5)
                new_height = driver.execute_script("return document.body.scrollHeight")
                if new_height == last_height:
                    break
                last_height = new_height

            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')

            category_name_tag = soup.find('h1')
            category_name = category_name_tag.text.strip() if category_name_tag else "unknown"

            products_from_category = parse_products_from_page(soup, category_url, category_name)

            if products_from_category:
                all_products.extend(products_from_category)
                logging.info(f"Собрано {len(products_from_category)} товаров из категории '{category_name}'")
            else:
                logging.warning(f"Для категории {category_url} не найдено товаров даже после ожидания.")

            # Промежуточное сохранение
            with open(RAW_PRODUCTS_FILE, "w", encoding="utf-8") as f:
                json.dump(all_products, f, ensure_ascii=False, indent=2)

    except Exception as e:
        logging.error(f"Произошла критическая ошибка: {e}")
    finally:
        logging.info("Сбор завершен. Не забудьте вручную закрыть браузер, запущенный для отладки.")

    # Финальная обработка и сохранение
    if all_products:
        unique_products = list({p['link']: p for p in all_products if p.get('link')}.values())
        logging.info(f"Итого собрано {len(unique_products)} уникальных товаров.")
        with open(RAW_PRODUCTS_FILE, "w", encoding="utf-8") as f:
            json.dump(unique_products, f, ensure_ascii=False, indent=2)
        logging.info(f"Финальные данные сохранены в {RAW_PRODUCTS_FILE}")
else:
    logging.error("Не удалось запустить сбор. Проверьте, запущен ли браузер и загружен ли список URL.")

2025-09-21 15:30:40,150 - INFO - Успешно загружено 105 URL категорий из файла.
2025-09-21 15:30:40,151 - INFO - Попытка подключения к существующему экземпляру Chrome на порту 9222...
2025-09-21 15:30:40,736 - INFO - Успешно подключено к существующему браузеру.
2025-09-21 15:30:40,769 - INFO - Обнаружен существующий файл. Загружено 9947 товаров.


Обход категорий:   0%|          | 0/105 [00:00<?, ?it/s]

2025-09-21 15:30:40,787 - INFO - Категория /catalog/apteka--251C13027/ уже обработана. Пропускаем.
2025-09-21 15:30:40,788 - INFO - Категория /catalog/aromaty-dlya-doma--251C13022/ уже обработана. Пропускаем.
2025-09-21 15:30:40,789 - INFO - Категория /catalog/bez-laktozy-khalyal--251C13041/ уже обработана. Пропускаем.
2025-09-21 15:30:40,790 - INFO - Категория /catalog/bumaga-i-salfetki--251C13026/ уже обработана. Пропускаем.
2025-09-21 15:30:40,790 - INFO - Категория /catalog/chay-kofe-kakao--251C13057/ уже обработана. Пропускаем.
2025-09-21 15:30:40,791 - INFO - Категория /catalog/chipsy-sukhariki--251C13064/ уже обработана. Пропускаем.
2025-09-21 15:30:40,792 - INFO - Категория /catalog/chistyashchie-sredstva--251C13020/ уже обработана. Пропускаем.
2025-09-21 15:30:40,792 - INFO - Категория /catalog/dacha-i-otdykh--251C12943/ уже обработана. Пропускаем.
2025-09-21 15:30:40,792 - INFO - Категория /catalog/dekor-dlya-doma--251C12945/ уже обработана. Пропускаем.
2025-09-21 15:30:40,79